# Lab 1: Predict Spark job failures with pandas and scikit-learn

**Phase 1, weeks 1-2.** You will explore a dataset of Spark job runs, then train two models that predict whether a job will fail.

The data is generated in the first cell, so nothing needs downloading. It mimics what you see in Databricks job history: cluster size, input volume, shuffle, skew, retries and the outcome.

**How to use this notebook:** run each cell with `Shift+Enter`, read the output, then do the **Your turn** tasks before moving on. Commit the notebook to GitHub when you finish.

In [ ]:
# Setup: run once. If an import fails, run in a terminal:  uv add pandas scikit-learn matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Ready")

## Step 1: Generate the job-run dataset

In [ ]:
rng = np.random.default_rng(42)
n = 5000
df = pd.DataFrame({
    "workers": rng.choice([2, 4, 8, 16, 32], n),
    "input_gb": np.round(rng.lognormal(3, 1.0, n), 1),
    "shuffle_gb": np.round(rng.lognormal(2, 1.2, n), 1),
    "skew_ratio": np.round(rng.gamma(2, 1.5, n), 2),        # largest partition / median partition
    "spot_instances": rng.choice([0, 1], n, p=[0.6, 0.4]),
    "job_type": rng.choice(["etl", "ml_training", "streaming", "reporting"], n, p=[0.5, 0.15, 0.15, 0.2]),
    "retries_last_7d": rng.poisson(0.5, n),
})
# Hidden rule that makes jobs fail: memory pressure, skew, spot interruptions, flaky history
gb_per_worker = (df.input_gb + df.shuffle_gb) / df.workers
logit = (-4 + 0.08 * gb_per_worker + 0.35 * df.skew_ratio + 0.9 * df.spot_instances
         + 0.6 * df.retries_last_7d + 0.7 * (df.job_type == "ml_training"))
df["failed"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
df.head()

## Step 2: Explore the data (EDA)

As a data engineer you already do this: check shape, nulls, distributions, and the target balance.

In [ ]:
print(df.shape)
print(df.isna().sum().sum(), "missing values")
print(f"Failure rate: {df.failed.mean():.1%}")
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df.groupby("job_type").failed.mean().sort_values().plot.barh(ax=axes[0], title="Failure rate by job type")
df.boxplot(column="skew_ratio", by="failed", ax=axes[1]); axes[1].set_title("Skew ratio vs failed")
df.groupby("workers").failed.mean().plot.bar(ax=axes[2], title="Failure rate by workers")
plt.suptitle(""); plt.tight_layout(); plt.show()

**Your turn 1:** add a column `gb_per_worker = (input_gb + shuffle_gb) / workers` to `df` and plot its failure rate in 10 buckets with `pd.qcut`. Does the failure rate rise with memory pressure?

In [ ]:
# Your code here

## Step 3: Split the data

The model learns from the training set and is judged on the test set it has never seen. `stratify` keeps the same failure rate in both.

In [ ]:
features = ["workers", "input_gb", "shuffle_gb", "skew_ratio", "spot_instances", "job_type", "retries_last_7d"]
X, y = df[features], df.failed
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(len(X_train), "train rows,", len(X_test), "test rows")

## Step 4: Train a baseline (logistic regression)

A `Pipeline` is like a Spark ML pipeline: preprocessing and model are one object, so the same steps run in training and in production.

In [ ]:
numeric = ["workers", "input_gb", "shuffle_gb", "skew_ratio", "spot_instances", "retries_last_7d"]
categorical = ["job_type"]
prep = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])
logreg = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000))])
logreg.fit(X_train, y_train)
proba = logreg.predict_proba(X_test)[:, 1]
print(f"ROC AUC: {roc_auc_score(y_test, proba):.3f}")
print(classification_report(y_test, logreg.predict(X_test), digits=3))

**Read the report:** *precision* = of the jobs we flagged, how many really failed. *Recall* = of the jobs that failed, how many we caught. For an on-call alert, which matters more to you?

## Step 5: Train a random forest and compare

In [ ]:
forest = Pipeline([("prep", prep), ("model", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))])
for name, model in [("Logistic regression", logreg), ("Random forest", forest)]:
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
    print(f"{name:20s} ROC AUC {scores.mean():.3f} ± {scores.std():.3f}")
forest.fit(X_train, y_train)
ConfusionMatrixDisplay.from_estimator(forest, X_test, y_test, display_labels=["succeeded", "failed"])
plt.show()

## Step 6: What drives failures?

In [ ]:
names = forest.named_steps["prep"].get_feature_names_out()
importance = pd.Series(forest.named_steps["model"].feature_importances_, index=names).sort_values()
importance.plot.barh(figsize=(7, 4), title="Feature importance (random forest)")
plt.tight_layout(); plt.show()

**Your turn 2:** add your `gb_per_worker` feature to `numeric`, retrain both models, and record the new ROC AUC.

**Your turn 3:** change the decision threshold. Use `proba > 0.3` instead of the default 0.5 and compare recall. When would an operations team prefer that?

## Wrap up

1. Write 3 lines in your learning log (`weeks/` file): what you built, the best ROC AUC, and one thing that surprised you.
2. Commit and push:

```powershell
git add .
git commit -m "Lab 1: Spark job failure classifier"
git push
```